In [ ]:
# GPT code: 

import numpy as np
import pandas as pd

def compute_cost(prefix_sum, prefix_count, i, j):
    """
    Compute negative log-likelihood cost for bucket [i, j)
    """
    total = prefix_count[j] - prefix_count[i]
    defaults = prefix_sum[j] - prefix_sum[i]

    if total == 0:
        return 0

    p = defaults / total

    # Avoid log(0)
    eps = 1e-10
    p = np.clip(p, eps, 1 - eps)

    cost = -(defaults * np.log(p) + (total - defaults) * np.log(1 - p))
    return cost


def optimal_binning(fico, default, K):
    """
    Optimal binning using dynamic programming
    
    Parameters:
        fico: array-like of FICO scores
        default: array-like of 0/1 labels
        K: number of buckets
        
    Returns:
        boundaries: list of split points
        assignments: bucket index for each observation
    """

    # Step 1: sort data
    df = pd.DataFrame({"fico": fico, "default": default})
    df = df.sort_values("fico").reset_index(drop=True)

    x = df["fico"].values
    y = df["default"].values
    n = len(x)

    # Step 2: prefix sums
    prefix_sum = np.zeros(n + 1)
    prefix_count = np.zeros(n + 1)

    for i in range(n):
        prefix_sum[i + 1] = prefix_sum[i] + y[i]
        prefix_count[i + 1] = prefix_count[i] + 1

    # Step 3: DP table
    dp = np.full((K + 1, n + 1), np.inf)
    prev = np.zeros((K + 1, n + 1), dtype=int)

    dp[0][0] = 0

    # Step 4: fill DP
    for k in range(1, K + 1):
        for j in range(1, n + 1):
            for i in range(j):
                cost = dp[k - 1][i] + compute_cost(prefix_sum, prefix_count, i, j)
                if cost < dp[k][j]:
                    dp[k][j] = cost
                    prev[k][j] = i

    # Step 5: backtrack to find splits
    boundaries_idx = []
    j = n
    for k in range(K, 0, -1):
        i = prev[k][j]
        boundaries_idx.append(i)
        j = i

    boundaries_idx = sorted(boundaries_idx)

    # Convert indices to FICO cutoffs
    boundaries = [x[idx] for idx in boundaries_idx if idx < n]

    # Step 6: assign buckets
    assignments = np.zeros(n, dtype=int)
    current_bucket = 0
    boundary_ptr = 0

    for i in range(n):
        if boundary_ptr < len(boundaries_idx) and i >= boundaries_idx[boundary_ptr]:
            current_bucket += 1
            boundary_ptr += 1
        assignments[i] = current_bucket

    df["bucket"] = assignments

    return boundaries, df